# Malawi VACS 2013 — PUD exploration

**Male** and **Female** PUDs are **separate** **`_.dta`** files under **`data/raw/Malawi Stata/`**. They share the **same variable names** (780 columns each). **Split-sample design:** female and male interviews were fielded in **different EAs** (see **`MALAWI_VACS_2013_DataUserGuide.pdf`**).

**Survey design (User Guide):** stratification variable **`REG`**, cluster variable **`PSU`**, final weight **`Finalwgt`**. PSUs are drawn from the **2010 MDHS** frame.

There is **no `SEX` column**—sex is **implicit** in which file a row comes from.

**Flow:** §1 Load both → §2 column list & EDA (male; schema matches female) → §3 samples & slot summaries → §4 harmonized TSV.


In [6]:
from pathlib import Path

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

COUNTRY_DIR = ROOT / "data" / "raw" / "Malawi Stata"
MALE_DTA = COUNTRY_DIR / "MALAWI_VACS_2013_Male_PUD.dta"
FEMALE_DTA = COUNTRY_DIR / "MALAWI_VACS_2013_Female_PUD.dta"
READ_KW = {}  # try {"encoding": "latin1"} if decode fails

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

`read_dta(..., **READ_KW)` for each file.


In [7]:
for p in (MALE_DTA, FEMALE_DTA):
    if not p.is_file():
        raise FileNotFoundError(p)

df_m, meta_m = pyreadstat.read_dta(MALE_DTA, **READ_KW)
df_f, meta_f = pyreadstat.read_dta(FEMALE_DTA, **READ_KW)

print(f"Male:   {MALE_DTA.name}  →  {df_m.shape[0]:,} × {df_m.shape[1]:,}")
print(f"Female: {FEMALE_DTA.name}  →  {df_f.shape[0]:,} × {df_f.shape[1]:,}")

assert list(df_m.columns) == list(df_f.columns), "Column order differs — reconcile before harmonizing"

def _resp_key(df: pd.DataFrame) -> pd.Series:
    return (
        df["PSU"].astype(int).astype(str)
        + "_"
        + df["EA"].astype(int).astype(str)
        + "_"
        + df["HH"].astype(int).astype(str)
    )

df_m = df_m.copy()
df_f = df_f.copy()
df_m["RESP_KEY"] = _resp_key(df_m)
df_f["RESP_KEY"] = _resp_key(df_f)

for tag, df in [("M", df_m), ("F", df_f)]:
    print(
        f"{tag} RESP_KEY unique / rows: {df['RESP_KEY'].nunique():,} / {len(df):,}  |  "
        f"dup rows (all cols): {int(df.duplicated().sum())}  |  "
        f"ID_DHS unique: {df['ID_DHS'].nunique()}"
    )

print(
    "District (DIST × District_name) sets: male",
    df_m.groupby(["DIST", "District_name"]).ngroups,
    "female",
    df_f.groupby(["DIST", "District_name"]).ngroups,
)

df_m[["RESP_KEY", "REG", "DIST", "District_name", "TA", "TA_name", "PSU", "EA", "HH", "ID_DHS", "Finalwgt"]].head(3)


Male:   MALAWI_VACS_2013_Male_PUD.dta  →  1,133 × 780
Female: MALAWI_VACS_2013_Female_PUD.dta  →  1,029 × 780
M RESP_KEY unique / rows: 1,133 / 1,133  |  dup rows (all cols): 0  |  ID_DHS unique: 122
F RESP_KEY unique / rows: 1,029 / 1,029  |  dup rows (all cols): 0  |  ID_DHS unique: 88
District (DIST × District_name) sets: male 26 female 27


,RESP_KEY,REG,DIST,District_name,TA,TA_name,PSU,EA,HH,ID_DHS,Finalwgt
0,5001_1_10,1.0,101.0,Chitipa,10101.0,Mwabulambya,5001.0,1.0,10.0,1.0,1648.272877
1,5001_1_15,1.0,101.0,Chitipa,10101.0,Mwabulambya,5001.0,1.0,15.0,1.0,1648.272877
2,5001_1_27,1.0,101.0,Chitipa,10101.0,Mwabulambya,5001.0,1.0,27.0,1.0,824.136438


In [10]:
df_f['RESP_KEY']

0         2001_55_2
1        2001_55_11
2       2001_55_101
3       2001_55_103
4       2001_55_112
           ...     
1024      2088_2_66
1025      2088_2_77
1026      2088_2_84
1027      2088_2_91
1028     2088_2_124
Name: RESP_KEY, Length: 1029, dtype: str

## 2. Column list & quick EDA

Full **`var_table`** on **male** PUD (female has identical columns). Compare **missingness** if needed by repeating with `df_f`.


In [9]:
df = df_m
meta = meta_m
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}

var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})
print(f"Variables: {len(df.columns):,}  |  Observations (male): {len(df):,}")
display(var_table.head(40))
display(var_table.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))
df.info(max_cols=18)


Variables: 781  |  Observations (male): 1,133


,column,stata_label,dtype,missing_n,missing_pct
REG,REG,REGION,float64,0,0.00
DIST,DIST,DISTRICT,float64,0,0.00
TA,TA,TA,float64,0,0.00
EA,EA,EA,float64,0,0.00
HH,HH,HOUSEHOLD,float64,0,0.00
ID_DHS,ID_DHS,ID_DHS,float64,0,0.00
NTOT,NTOT,Number of people living in this household,float64,0,0.00
NSEL,NSEL,Number of females in this household,float64,0,0.00
LINE_01,LINE_01,Household Member Line(1),float64,0,0.00
LINE_02,LINE_02,Household Member Line(2),float64,24,2.12


,column,stata_label,dtype,missing_n,missing_pct
0,Q407,407. How old were you the first time that you got pregnant?,float64,1133,100.0
1,AGE_20,Age of Household Member(20),float64,1133,100.0
2,Q422,"422. Was this person was more than 10 years older than you, 5-10 yea...",float64,1133,100.0
3,Q905,"905. This last time, how many people physically forced you to have sex?",float64,1133,100.0
4,RES_20,Household Member live in this house(20),float64,1133,100.0
5,Q408,408. Have you ever had a pregnancy that did not end in a live birth?,float64,1133,100.0
6,ORDER_08,ELEGIBLE RESPONDEMENT(8),float64,1133,100.0
7,ORDER_09,ELEGIBLE RESPONDEMENT(9),float64,1133,100.0
8,ORDER_10,ELEGIBLE RESPONDEMENT(10),float64,1133,100.0
9,ORDER_11,ELEGIBLE RESPONDEMENT(11),float64,1133,100.0


<class 'pandas.DataFrame'>
RangeIndex: 1133 entries, 0 to 1132
Columns: 781 entries, REG to RESP_KEY
dtypes: float64(510), str(271)
memory usage: 6.8 MB


## 3. Further EDA and exploration

### Raw row samples


In [11]:
pd.set_option("display.max_columns", 42)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 72)

_core = [
    "RESP_KEY", "REG", "DIST", "District_name", "TA", "TA_name",
    "PSU", "EA", "HH", "ID_DHS", "Finalwgt", "Q1_HH", "Q1_MM", "NTOT",
]
display(df_m[_core].head(6))
display(df_f[_core].head(6))
display(df_m[_core].sample(4, random_state=1))
display(df_f[_core].sample(4, random_state=2))


,RESP_KEY,REG,DIST,District_name,TA,TA_name,PSU,EA,HH,ID_DHS,Finalwgt,Q1_HH,Q1_MM,NTOT
0,5001_1_10,1.0,101.0,Chitipa,10101.0,Mwabulambya,5001.0,1.0,10.0,1.0,1648.272877,10.0,25.0,3.0
1,5001_1_15,1.0,101.0,Chitipa,10101.0,Mwabulambya,5001.0,1.0,15.0,1.0,1648.272877,12.0,50.0,10.0
2,5001_1_27,1.0,101.0,Chitipa,10101.0,Mwabulambya,5001.0,1.0,27.0,1.0,824.136438,13.0,0.0,5.0
3,5002_17_1,1.0,101.0,Chitipa,10101.0,Mwabulambya,5002.0,17.0,1.0,3.0,1685.929995,9.0,34.0,7.0
4,5002_17_25,1.0,101.0,Chitipa,10101.0,Mwabulambya,5002.0,17.0,25.0,3.0,1123.953330,13.0,14.0,5.0
5,5003_38_3,1.0,101.0,Chitipa,10101.0,Mwabulambya,5003.0,38.0,3.0,6.0,755.163863,15.0,30.0,5.0


,RESP_KEY,REG,DIST,District_name,TA,TA_name,PSU,EA,HH,ID_DHS,Finalwgt,Q1_HH,Q1_MM,NTOT
0,2001_55_2,1.0,101.0,Chitipa,10101.0,Mwabulambya,2001.0,55.0,2.0,9.0,955.435932,15.0,6.0,7.0
1,2001_55_11,1.0,101.0,Chitipa,10101.0,Mwabulambya,2001.0,55.0,11.0,9.0,955.435932,12.0,3.0,4.0
2,2001_55_101,1.0,101.0,Chitipa,10101.0,Mwabulambya,2001.0,55.0,101.0,9.0,955.435932,13.0,48.0,5.0
3,2001_55_103,1.0,101.0,Chitipa,10101.0,Mwabulambya,2001.0,55.0,103.0,9.0,955.435932,14.0,24.0,5.0
4,2001_55_112,1.0,101.0,Chitipa,10101.0,Mwabulambya,2001.0,55.0,112.0,9.0,582.117977,8.0,59.0,2.0
5,2001_55_113,1.0,101.0,Chitipa,10101.0,Mwabulambya,2001.0,55.0,113.0,9.0,1164.235954,14.0,14.0,10.0


,RESP_KEY,REG,DIST,District_name,TA,TA_name,PSU,EA,HH,ID_DHS,Finalwgt,Q1_HH,Q1_MM,NTOT
453,5024_30_27,2.0,202.0,Nkhota kota,20203.0,Malengachanzi,5024.0,30.0,27.0,194.0,280.019229,14.0,50.0,3.0
335,5054_712_49,2.0,208.0,Dedza,20820.0,Dedza Town,5054.0,712.0,49.0,403.0,287.388686,10.0,8.0,4.0
912,5121_12_30,3.0,315.0,Blantyre,31546.0,Bangwe Ward,5121.0,12.0,30.0,596.0,771.487745,15.0,22.0,5.0
94,5004_71_28,1.0,101.0,Chitipa,10101.0,Mwabulambya,5004.0,71.0,28.0,11.0,4334.685085,14.0,1.0,6.0


,RESP_KEY,REG,DIST,District_name,TA,TA_name,PSU,EA,HH,ID_DHS,Finalwgt,Q1_HH,Q1_MM,NTOT
549,2047_8_549,2.0,210.0,Lilongwe,21087.0,Area 57,2047.0,8.0,549.0,342.0,1491.854986,10.0,3.0,7.0
987,2085_3_40,3.0,315.0,Blantyre,31540.0,Chilomoni Ward,2085.0,3.0,40.0,589.0,328.201328,16.0,4.0,4.0
486,2042_32_83,2.0,209.0,Ntcheu,20904.0,S/C Makwangwala,2042.0,32.0,83.0,414.0,746.243967,9.0,32.0,5.0
580,2049_58_154,3.0,301.0,Mangochi,30101.0,Mponda,2049.0,58.0,154.0,437.0,13407.392506,14.0,23.0,5.0


### Slot summaries (ID / geo / design)

**Male** file; **female** uses the **same** names (counts differ).


In [12]:
df = df_m
L = meta_m.column_names_to_labels or {}

_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title: str, cols: list, note_extra: str = ""):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (str(L.get(c) or ""))[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Respondent row key (derived)", ["RESP_KEY"], "not a released PUD_ID; PSU+EA+HH unique per row in each file")
slot_summary("2. ID_DHS (MDHS linkage, not person-unique)", ["ID_DHS"], "many respondents share the same ID_DHS — see §4")
slot_summary("3. Region (stratum in User Guide)", ["REG"], "User Guide: stratification variable")
slot_summary("4. District code + name", ["DIST", "District_name"], "")
slot_summary("5. TA + name", ["TA", "TA_name"], "")
slot_summary(
    "6. Enumeration area + household",
    ["EA", "HH"],
    "HH: integer household index within EA; **1–3 digits** as int string (values 1–966 male, 1–841 female in these PUDs)",
)
slot_summary("7. Cluster (User Guide)", ["PSU"], "User Guide: cluster variable for complex surveys")
slot_summary("8. Final weight", ["Finalwgt"], "")
slot_summary("9. Interview start (respondent)", ["Q1_HH", "Q1_MM"], "H1_* has slightly more missing than Q1_* on some rows")
slot_summary("10. Roster / HH size", ["NTOT", "NSEL"], "")



1. Respondent row key (derived)
  RESP_KEY | 
    8-12 chars (string); no male/female text (heuristic); dtype=str; n_distinct=1133; missing=0
   not a released PUD_ID; PSU+EA+HH unique per row in each file

2. ID_DHS (MDHS linkage, not person-unique)
  ID_DHS | ID_DHS
    1-3 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1.0/807.0; n_distinct=122; missing=0
   many respondents share the same ID_DHS — see §4

3. Region (stratum in User Guide)
  REG | REGION
    1 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=1.0/3.0; n_distinct=3; missing=0
   User Guide: stratification variable

4. District code + name
  DIST | DISTRICT
    3 digits (integer codes); no M/F identifier (numeric); dtype=float64; min/max=101.0/315.0; n_distinct=26; missing=0
  District_name | District_name
    4-11 chars (string); no male/female text (heuristic); dtype=str; n_distinct=25; missing=0

5. TA + name
  TA | TA
    5 digits (integer codes); no M/F iden

## 4. Harmonized codebook slots (Malawi 2013)

**Excel:** **one row per country / wave** per slot; **`variable_male`** and **`variable_female`** list the **same Stata names** here because both PUDs use **identical schemas**. Note **split-sample EAs** (male vs female files) in **`notes`**.

**Sources:** `MALAWI_VACS_2013_Male_PUD.dta`, `MALAWI_VACS_2013_Female_PUD.dta`. **PDFs:** **`MALAWI_VACS_2013_DataUserGuide.pdf`** (design: **REG** / **PSU** / **Finalwgt**), **`MALAWI_VACS_2013_DataFileAppendix.pdf`** (data-quality notes, `_OT` text vars).

### Respondent / household keys

- **No** released person-level ID column. **`RESP_KEY` = `PSU` + `_` + `EA` + `_` + `HH`** (integer string form) is **unique within each file** and matches one sampled adolescent per household.
- **`ID_DHS`:** ties to the **MDHS / census** frame; **not unique** across rows—**do not** treat as respondent ID.

```
slot	variable_male	variable_female	type_and_width	notes
Respondent row key	(derived) str(PSU)+'_'+str(EA)+'_'+str(HH)	(same)	**8–12 chars** (underscores; digit runs vary)	Unique **within** each PUD; **not** a published `VACS_ID`-style field; male and female samples use **disjoint** EAs
Household number	HH	HH	**`float64`** whole-number codes; **1–3 digits** if printed as integers (**1–966** male PUD, **1–841** female PUD)	Household index **within EA** (not global); interpret with **PSU** + **EA**; one adolescent respondent per household. **Not** a fixed-width string column—width is the **integer string length** (1–3 here)
Geo level 1	REG	REG	1 digit (1–3)	Stata label **REGION**; User Guide: **stratification** (North / Central / South)
Geo level 2	DIST District_name	DIST District_name	numeric **DIST** + string **District_name**	**Asymmetry:** district **combinations present** differ by file (split sample — e.g. male **Phalombe** vs female **Karonga** / **Neno** in this extract)
Geo level 3	TA TA_name	TA TA_name	numeric **TA** + string **TA_name**	Traditional authority
EA	EA	EA	integer	Enumeration area (within **PSU**); not the **svy cluster** variable per User Guide
Cluster (svy)	PSU	PSU	integer	User Guide lists **PSU** as the **cluster** variable for analysis
Stratum (svy)	REG	REG	(same as Geo level 1)	User Guide lists **Reg** as **stratification** variable (with **PSU**, **Finalwgt**)
Weight	Finalwgt	Finalwgt	float	Post-stratified **final weight** (three-step procedure in User Guide)
Sex	—	—	—	**No `SEX` column**; **Male** file = males, **Female** file = females
Interview start	Q1_HH Q1_MM	Q1_HH Q1_MM	hour **1–22**, minute **0–59** (numeric)	Stata labels: interview began; **Q1_** has full respondent coverage; **H1_** alternative with slightly more missing in this extract
DHS linkage	ID_DHS	ID_DHS	float (numeric code)	**Not row-unique** — shared MDHS / design linkage, **not** a person key
```
